# 1. Objective

This notebook explores **three prompting techniques** — Direct Prompting, Chain-of-Thought (CoT), and Tree-of-Thought (ToT) — applied to **social media brand monitoring** for OpenAI.

**Technique:** Prompt engineering strategies for LLM-based sentiment and brand analysis

**Why relevant to social media brand monitoring:**
Reddit posts provide unfiltered, real-time public opinion about a brand. Choosing the right prompting strategy directly affects the quality of sentiment labels, depth of reasoning, and actionability of brand insights extracted from social media data. Comparing Direct, CoT, and ToT prompting helps identify which approach produces the most accurate, explainable, and useful output for brand analysts monitoring OpenAI's public perception.

# 2. Setup

Install dependencies and load the sample dataset.

In [ ]:
!pip install openai pandas tqdm matplotlib

In [ ]:
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from openai import OpenAI
from google.colab import userdata, files

In [ ]:
# Initialise OpenAI client
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
print("Connected Successfully")

In [ ]:
# Upload and load the dataset (JSONL format — one Reddit post per line)
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

data = []
with open(file_name, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)
df.head()

In [ ]:
df.info()
df[["title", "text", "subreddit", "score", "num_comments"]].head()

### Dataset Overview

The dataset contains Reddit posts related to OpenAI. Each row represents an individual post collected from OpenAI-related discussions and communities.

Key fields used in this analysis:

- **Title** – The headline or main topic of the Reddit post.
- **Text** – The content or body of the post.
- **Subreddit** – The Reddit community where the post was published.
- **Score** – Upvotes received, indicating community engagement and popularity.
- **Number of Comments** – Total user responses to the post.
- **Top Comments** – Highly ranked comments providing additional context and opinions.

These fields are valuable for analysing public opinion, sentiment, engagement patterns, and discussion trends related to OpenAI.

In [ ]:
# Filter short posts and draw a reproducible sample of 20 posts
df["combined_text"] = df["title"].fillna("") + " " + df["text"].fillna("")
df = df[df["combined_text"].str.len() > 20].copy()

sample_df = df.sample(20, random_state=42).reset_index(drop=True)
sample_df[["title", "subreddit", "score", "num_comments"]].head()

# 3. Implementation

Three prompting strategies are implemented below:

| Strategy | Description |
|---|---|
| **Direct Prompting** | One-shot instruction — ask for sentiment, topic, explanation, and brand insight directly |
| **Chain-of-Thought (CoT)** | Structured step-by-step reasoning through topic, evidence, sentiment, motivation, and insight |
| **Tree-of-Thought (ToT)** | Multi-branch reasoning — explores positive, negative, and mixed interpretations before converging |

In [ ]:
def direct_prompt(post_text: str) -> str:
    """
    Direct prompting: single-turn instruction asking for sentiment,
    topic, explanation, and brand insight.

    Args:
        post_text: Combined title + body text of a Reddit post.

    Returns:
        LLM response as a plain string.
    """
    prompt = f"""
Analyse the following Reddit post about OpenAI.

Post:
{post_text}

Return:
1. Sentiment: Positive, Negative, Neutral, or Mixed
2. Main topic
3. Short explanation
4. Brand insight for OpenAI
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

In [ ]:
def cot_structured_prompt(post_text: str) -> str:
    """
    Chain-of-Thought prompting: structured six-step reasoning through
    topic identification, evidence, sentiment, user motivation, brand
    insight, and a final summary.

    Args:
        post_text: Combined title + body text of a Reddit post.

    Returns:
        LLM response as a plain string.
    """
    prompt = f"""
Analyse the following Reddit post about OpenAI using a structured reasoning approach.

Post:
{post_text}

Return the answer in this structure:

1. Topic Identification:
What is the post mainly about?

2. Evidence from the Post:
What words, phrases, or ideas support the interpretation?

3. Sentiment Analysis:
Classify the sentiment as Positive, Negative, Neutral, or Mixed.

4. User Motivation:
What might the Reddit user be feeling or trying to express?

5. Brand/Business Insight:
What can OpenAI learn from this post?

6. Final Summary:
Give a short final interpretation.
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

In [ ]:
def tree_of_thought_prompt(post_text: str) -> str:
    """
    Tree-of-Thought prompting: explores three sentiment branches
    (positive, negative, mixed/neutral), evaluates each, then converges
    on the strongest interpretation with a final brand insight.

    Args:
        post_text: Combined title + body text of a Reddit post.

    Returns:
        LLM response as a plain string.
    """
    prompt = f"""
Analyse the following Reddit post about OpenAI using a Tree-of-Thought approach.

Post:
{post_text}

Consider three possible interpretations:

Interpretation A: The post is mainly positive.
Interpretation B: The post is mainly negative.
Interpretation C: The post is mixed or neutral.

For each interpretation:
1. Explain why it could be correct.
2. Give evidence from the post.
3. Identify any weakness in that interpretation.

Then choose the strongest final interpretation.

Return:
1. Interpretation A
2. Interpretation B
3. Interpretation C
4. Best final interpretation
5. Final sentiment
6. Brand insight for OpenAI
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content

# 4. Results

Run all three prompting methods across the 20-post sample and collect outputs, timings, and output lengths.

In [ ]:
results = []

for i, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    post_text = row["combined_text"]

    start = time.time()
    direct_result = direct_prompt(post_text)
    direct_time = time.time() - start

    start = time.time()
    cot_result = cot_structured_prompt(post_text)
    cot_time = time.time() - start

    start = time.time()
    tot_result = tree_of_thought_prompt(post_text)
    tot_time = time.time() - start

    results.append({
        "post_id":       row["post_id"],
        "title":         row["title"],
        "subreddit":     row["subreddit"],
        "score":         row["score"],
        "num_comments":  row["num_comments"],
        "direct_result": direct_result,
        "direct_time":   direct_time,
        "cot_result":    cot_result,
        "cot_time":      cot_time,
        "tot_result":    tot_result,
        "tot_time":      tot_time,
    })

results_df = pd.DataFrame(results)
results_df.head()

In [ ]:
# Example outputs on brand/social media data — first 3 posts
for i in range(3):
    post = sample_df.iloc[i]["combined_text"]
    print("\n" + "=" * 100)
    print(f"POST {i+1}: {sample_df.iloc[i]['title']}")
    print("=" * 100)
    print(post[:500])

    print("\n--- DIRECT PROMPT ---")
    print(results_df.iloc[i]["direct_result"])

    print("\n--- CHAIN-OF-THOUGHT ---")
    print(results_df.iloc[i]["cot_result"])

    print("\n--- TREE-OF-THOUGHT ---")
    print(results_df.iloc[i]["tot_result"])

In [ ]:
# Metrics table
results_df["direct_length"] = results_df["direct_result"].str.len()
results_df["cot_length"]    = results_df["cot_result"].str.len()
results_df["tot_length"]    = results_df["tot_result"].str.len()

metrics = pd.DataFrame({
    "Method": ["Direct Prompting", "Chain-of-Thought", "Tree-of-Thought"],
    "Avg Runtime (s)": [
        results_df["direct_time"].mean(),
        results_df["cot_time"].mean(),
        results_df["tot_time"].mean()
    ],
    "Avg Output Length (chars)": [
        results_df["direct_length"].mean(),
        results_df["cot_length"].mean(),
        results_df["tot_length"].mean()
    ]
})

metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(metrics["Method"], metrics["Avg Runtime (s)"])
axes[0].set_title("Average Runtime Comparison")
axes[0].set_xlabel("Prompting Method")
axes[0].set_ylabel("Average Runtime (seconds)")
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(metrics["Method"], metrics["Avg Output Length (chars)"])
axes[1].set_title("Average Output Length Comparison")
axes[1].set_xlabel("Prompting Method")
axes[1].set_ylabel("Average Output Length (characters)")
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

# 5. Comparison

All three methods were evaluated on the same sample of 20 Reddit posts about OpenAI.

| Dimension | Direct Prompting | Chain-of-Thought | Tree-of-Thought |
|---|---|---|---|
| **Output depth** | Shallow — one label per field | Moderate — reasoned steps | Deep — multi-branch exploration |
| **Sentiment accuracy** | Can miss nuance | Stronger via evidence tracing | Strongest — considers alternatives |
| **Brand insight quality** | Generic | Contextualised | Most actionable |
| **Avg runtime** | Fastest | Moderate | Slowest |
| **Avg output length** | Shortest | Medium | Longest |
| **Cost** | Lowest token usage | Moderate | Highest token usage |
| **Suitability** | High-volume triage | Balanced use | Deep analysis on key posts |

# 6. Justification

**Why Chain-of-Thought is the recommended default for brand monitoring pipelines:**

Direct Prompting produces rapid output, but it lacks transparency. For brand monitoring, where analysts need to understand *why* a post is classified as negative before escalating to a PR response, a single-label answer is insufficient. CoT prompting forces the model to surface its reasoning at each step — topic identification, evidence, sentiment classification, and user motivation — giving brand teams an audit trail they can trust and act on.

Tree-of-Thought produces the highest-quality outputs, particularly for ambiguous or high-stakes posts, because it explicitly weighs competing interpretations before committing to a final answer. However, its longer runtime and higher token cost make it impractical as a default for monitoring hundreds of posts per day. It is best reserved for posts that exceed a score or engagement threshold, where investment in deeper analysis is justified.

A hybrid pipeline — CoT for routine monitoring, ToT for escalated high-engagement posts — provides the best balance of coverage, cost, and insight quality for real-world social media brand monitoring of OpenAI.

# 7. Limitations

All three prompting techniques share a fundamental limitation: they rely on the underlying language model (GPT-4o-mini) to self-assess sentiment rather than comparing against human-labelled ground truth. Reddit posts about OpenAI often contain irony, sarcasm, or highly technical criticism that is difficult to classify without domain expertise. Additionally, the techniques produce unstructured natural-language outputs, making downstream aggregation (e.g., calculating the percentage of posts that are negative this week vs. last week) non-trivial without a second parsing step. Tree-of-Thought in particular is sensitive to prompt wording — small changes can shift which branch the model treats as strongest, reducing reproducibility across prompt iterations.

# 8. Pipeline Connection

This notebook sits at the **analysis layer** of the brand monitoring pipeline. Upstream, `shared/data_loader.py` ingests raw Reddit posts via the Pushshift or Reddit API and writes them to the shared JSONL format consumed here. The output of this notebook — `results_df` — feeds downstream into the **aggregation and alerting layer**, where sentiment trends are tracked over rolling time windows, high-engagement negative posts trigger Slack/email alerts to the PR team, and topic clusters are passed to the reporting dashboard. The `analyse_post()` export function (Section 9) is the clean interface point between this notebook and the pipeline orchestrator (e.g., Airflow, Prefect), allowing the chosen prompting strategy to be swapped without changes to upstream or downstream components.

# 9. Export Function

Clean, type-hinted export function for use by Phase 2 of the pipeline.

In [ ]:
from typing import Literal


def analyse_post(
    post_text: str,
    method: Literal["direct", "cot", "tot"] = "cot",
) -> str:
    """
    Analyse a single Reddit post about OpenAI using the specified
    prompting strategy and return the model's raw analysis string.

    This is the pipeline-facing export function. The downstream
    aggregation layer calls this for each post and stores the result
    alongside post metadata for trend analysis and alerting.

    Args:
        post_text: Combined title and body text of a Reddit post.
            Typically constructed as ``title + " " + text``.
        method: Prompting strategy to use.
            - ``"direct"`` – fast, single-pass analysis (best for bulk triage).
            - ``"cot"``    – Chain-of-Thought structured reasoning (default;
              recommended for routine monitoring).
            - ``"tot"``    – Tree-of-Thought multi-branch analysis (best for
              high-engagement or ambiguous posts).

    Returns:
        Raw LLM analysis as a plain string containing sentiment label,
        topic, reasoning steps, and brand insights per the chosen method.

    Raises:
        ValueError: If ``method`` is not one of ``"direct"``, ``"cot"``,
            or ``"tot"``.

    Example::

        result = analyse_post(
            post_text="OpenAI just released GPT-5 and it's incredible!",
            method="cot"
        )
        print(result)
    """
    dispatch = {
        "direct": direct_prompt,
        "cot":    cot_structured_prompt,
        "tot":    tree_of_thought_prompt,
    }

    if method not in dispatch:
        raise ValueError(
            f"Unknown method '{method}'. Choose from: {list(dispatch.keys())}"
        )

    return dispatch[method](post_text)

In [ ]:
# Smoke test — verify the export function works end-to-end
test_post = sample_df.iloc[0]["combined_text"]
print(analyse_post(test_post, method="cot"))